# MGMT radiomics baseline pipeline (dummy → RF → GA-RF)

This notebook shows how to:
- load the four pre-split CSV files you created,
- get floor performance using dummy classifiers,
- train a plain Random Forest on all radiomics features,
- plug in a GA-selected feature subset to train GA-RF.
- make ROC curve and show result through hyperparameter tuning


In [1]:
import os
import random

import pandas as pd
import numpy as np

import xgboost as xgb

from sklearn.model_selection import cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import (
    StratifiedKFold,
    RepeatedStratifiedKFold,
    RandomizedSearchCV,
)
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve,
    auc,
    classification_report,
    confusion_matrix,
)

import matplotlib.pyplot as plt

import cuml
from cuml.ensemble import RandomForestClassifier



In [2]:
# !git clone https://github.com/rapidsai/rapidsai-csp-utils.git
# !python rapidsai-csp-utils/colab/pip-install.py


In [3]:
# conda create -n rapids -c rapidsai -c conda-forge -c nvidia python=3.10 rapids=25.10 cudatoolkit=12.4
# /home/jpark74/miniconda3/bin/conda init bash
# source ~/.bashrc

# conda activate rapids

## 1. Load data
We load `X_train_xgbsel.csv`, `X_test_xgbsel.csv`, `y_train.csv`, and `y_test.csv`.  
Labels are squeezed to 1D so that scikit-learn can use them directly.  
If your labels are strings like `"Methylated"`, you can map them to integers right after loading.


In [4]:
# set seeds for reproducibility
random.seed(42)
np.random.seed(42)

# read feature matrices
X_train = pd.read_csv("../data/X_train_xgbsel.csv")   # training features
X_test  = pd.read_csv("../data/X_test_xgbsel.csv")    # held-out test features

# read target vectors (squeeze → Series, not DataFrame)
y_train = pd.read_csv("../data/y_train.csv").squeeze()
y_test  = pd.read_csv("../data/y_test.csv").squeeze()
# DATA_DIR = "/content/drive/MyDrive/gbm-mgmt-radiomics/data"

# X_train = pd.read_csv(f"{DATA_DIR}/X_train_xgbsel.csv")
# X_test  = pd.read_csv(f"{DATA_DIR}/X_test_xgbsel.csv")
# y_train = pd.read_csv(f"{DATA_DIR}/y_train.csv").squeeze("columns")
# y_test  = pd.read_csv(f"{DATA_DIR}/y_test.csv").squeeze("columns")



# quick sanity check on shapes
print("X_train:", X_train.shape)   # (n_train, d)
print("X_test: ", X_test.shape)    # (n_test, d)
print("y_train:", y_train.shape)   # (n_train,)
print("y_test: ", y_test.shape)    # (n_test,)

# --- GA will select columns from TRAIN ---
New_FS = X_train.copy()               # GA works on training features
y_trn  = y_train.reset_index(drop=True)

print("GA base features (train):", New_FS.shape)
print("GA base labels  (train):", y_trn.shape)

GA_POP_PATH   = "../data/ga_pop.npy"
GA_SCORE_PATH = "../data/ga_scores.npy"
size  = 50
n_feat = New_FS.shape[1]   # number of features GA will see

# 1) Convert pandas DataFrames to NumPy arrays in advance
X_np = New_FS.to_numpy(dtype=np.float32)
y_np = y_trn.to_numpy()

# 2) Precompute the folds from StratifiedKFold (we’ll reuse these in fitness)
from sklearn.model_selection import StratifiedKFold
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = [(train_idx, test_idx) for train_idx, test_idx in kfold.split(X_np, y_np)]


# if labels are text, map to 0/1 here
# y_trn = (y_trn == "Methylated").astype(int)
# y_test = (y_test == "Methylated").astype(int)


X_train: (53, 38)
X_test:  (65, 38)
y_train: (53,)
y_test:  (65,)
GA base features (train): (53, 38)
GA base labels  (train): (53,)


In [ ]:
# if 'BASE_DIR' in globals() and BASE_DIR:
#     os.makedirs(BASE_DIR, exist_ok=True)

# # Collect checkpoint paths that are defined in the current runtime
# CANDIDATE_PATH_VARS = [
#      'PROG_PATH', 'GA_POP_PATH', 'GA_SCORE_PATH',
#     'GA_CHROMO_PATH', 'RNG_PATH', 'BEST_PATH'
# ]
# paths = []
# for var in CANDIDATE_PATH_VARS:
#     p = globals().get(var, None)
#     if isinstance(p, str) and p:
#         paths.append(p)

# # Delete checkpoint files (safe to run multiple times)
# for p in paths:
#     try:
#         os.remove(p)
#         print("removed:", p)
#     except FileNotFoundError:
#         pass  # it's fine if the file wasn't there

# # Clear in-memory variables so nothing leaks into the new run
# pop = None
# history_score, history_chromo = [], []

# # (Optional) Re-seed RNG for reproducibility
# np.random.seed(42)
# random.seed(42)

# # Force the resume cursor and timer to reset
# start_run, start_gen = 0, 0
# t0 = time.time()

# print("exists:", [os.path.exists(p) for p in [PROG_PATH, GA_POP_PATH, GA_SCORE_PATH, globals().get('GA_CHROMO_PATH',''), globals().get('RNG_PATH',''), BEST_PATH]])
# print("cursor:", start_run, start_gen)
# print("mem:", pop, len(history_score), len(history_chromo))

In [5]:
# from google.colab import drive
# drive.mount('/content/drive')

## 2. Dummy baselines
We train two dummy models to see the minimum performance on this split.  
`most_frequent` = always predict the majority class.  
`stratified` = predict classes according to training distribution (a bit harder baseline).


In [6]:
# dummy that always predicts the most frequent class in y_train
dummy_mf = DummyClassifier(strategy="most_frequent")
dummy_mf.fit(X_train, y_train)
y_pred_mf = dummy_mf.predict(X_test)
acc_mf = accuracy_score(y_test, y_pred_mf)
print(f"Dummy (most_frequent) accuracy on TEST: {acc_mf:.3f}")

# dummy that samples labels according to class proportion in y_train
dummy_st = DummyClassifier(strategy="stratified", random_state=42)
dummy_st.fit(X_train, y_train)
y_pred_st = dummy_st.predict(X_test)
acc_st = accuracy_score(y_test, y_pred_st)
print(f"Dummy (stratified) accuracy on TEST:   {acc_st:.3f}")

Dummy (most_frequent) accuracy on TEST: 0.200
Dummy (stratified) accuracy on TEST:   0.462


## 3. Plain Random Forest (no GA)

We now train a real model on **all 38 radiomics features** using 54 training cases.  
Because the training set is small and high-dimensional, we first run a 5-fold **stratified** cross-validation on the training set to get a more stable estimate.  



In [ ]:
# ---- User settings ----
SEED = 42
N_SPLITS = 5

# ---- Preconditions (asserts to fail fast) ----
assert "X_np" in globals() and "y_np" in globals(), "Please define X_np, y_np first."
assert "CURF" in globals(), "Please import/alias your RF class as CURF (cuML or sklearn)."
X_np = np.asarray(X_np)
y_np = np.asarray(y_np).ravel()

# ---- Fixed folds for reproducibility ----
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = list(cv.split(X_np, y_np))

# ---- Helpers ----
def _safe_auc_or_acc(y_true, y_proba=None, y_pred=None):
    """Prefer AUC if proba available; otherwise use accuracy."""
    try:
        if y_proba is not None:
            return roc_auc_score(y_true, y_proba)
    except Exception:
        pass
    if y_pred is None and y_proba is not None:
        y_pred = (y_proba >= 0.5).astype(int)
    return accuracy_score(y_true, y_pred)

def _cap_n_bins_for_folds(n_bins, folds):
    """For cuML: ensure n_bins ≤ min train size across folds (avoids auto-adjust)."""
    min_train = min(len(tr) for tr, _ in folds)
    return int(min(max(2, min_train), n_bins))

def _mean_cv_score(params, folds):
    """Train/eval with given params on fixed folds, return mean±std score."""
    scores = []

    # keep n_bins safe if provided (no harm for sklearn; ignored if unsupported)
    p = dict(params)
    if "n_bins" in p:
        p["n_bins"] = _cap_n_bins_for_folds(p["n_bins"], folds)

    for tr_idx, te_idx in folds:
        X_tr, X_te = X_np[tr_idx], X_np[te_idx]
        y_tr, y_te = y_np[tr_idx], y_np[te_idx]

        rf = CURF(
            n_estimators=p.get("n_estimators", 100),
            max_depth=p.get("max_depth", 12),
            max_features=p.get("max_features", "sqrt"),
            min_samples_leaf=p.get("min_samples_leaf", 2),
            bootstrap=p.get("bootstrap", True),
            random_state=SEED,
            **({ "n_bins": p["n_bins"] } if "n_bins" in p else {})
        )
        rf.fit(X_tr, y_tr)

        y_proba = None
        y_pred  = None
        try:
            y_proba = rf.predict_proba(X_te)[:, 1]
        except Exception:
            try:
                y_pred = rf.predict(X_te)
            except Exception:
                scores.append(0.0)
                continue

        scores.append(_safe_auc_or_acc(y_te, y_proba=y_proba, y_pred=y_pred))

    scores = np.asarray(scores, dtype=float)
    return float(scores.mean()), float(scores.std())

# ---- Search space (lightweight but effective for GA ranking) ----
param_space = {
    "n_estimators":     [100,150],              # keep light for GA evaluator
    "max_depth":        [10, 12],       # shallow-ish to stabilize ranking
    "max_features":     ["sqrt", 0.5, 0.7], # sqrt and fractions
    "min_samples_leaf": [1, 2, 4],
    "bootstrap":        [True],
    "n_bins":           [32],               # cuML only; sklearn will ignore via **kwargs guard
}

# ---- Grid search ----
keys   = list(param_space.keys())
combos = list(product(*[param_space[k] for k in keys]))

best = {"mean": -1.0, "std": 1e9, "params": None}
for vals in combos:
    params = dict(zip(keys, vals))
    m, s = _mean_cv_score(params, folds)
    print(f"[GA-evaluator] {params} -> mean={m:.4f}, std={s:.4f}")

    better = (m > best["mean"]) or (np.isclose(m, best["mean"]) and s < best["std"])
    if better:
        best = {"mean": m, "std": s, "params": params}

print("\n[Chosen GA evaluator params]")
print(best["params"], f"mean={best['mean']:.4f}, std={best['std']:.4f}")

XGB(GPU) 5-fold CV accuracy: [0.72727273 0.63636364 0.90909091 0.8        0.7       ]
XGB(GPU) 5-fold CV accuracy (mean): 0.7545454545454546
XGB(GPU) 5-fold CV AUC: [0.8        0.66666667 0.93333333 0.92       0.88      ]
XGB(GPU) 5-fold CV AUC (mean): 0.8400000000000001

XGB(GPU) TEST accuracy: 0.646
XGB(GPU) TEST AUC:      0.571

XGB(GPU) TEST classification report:
              precision    recall  f1-score   support

           0       0.19      0.23      0.21        13
           1       0.80      0.75      0.77        52

    accuracy                           0.65        65
   macro avg       0.49      0.49      0.49        65
weighted avg       0.67      0.65      0.66        65



## 4. GA-RF (paper-faithful version)

Below is a version that matches the 2022 code style much more closely.

Key points:

- we recreate their globals: `New_FS`, `y_trn`, `kfold`, `model`
- we keep their function names: `initilization_of_population`, `fitness_score`, `selection`, `crossover`, `mutation`, `generations`
- we fix only the things that would break in pandas/NumPy now:
  - `np.bool` → `bool`
  - `.iloc[train].iloc[:,chromosome]` → `.iloc[train, chromosome]`
- at the end we **actually call** `generations(...)` so you see `gen 0 ...`, `gen 1 ...` printed like their script
- this version uses **your** data: `X_train` → `New_FS`, `y_train` → `y_trn`

You can change `n_gen` or `size` later. After this, in step 5, you can take `best_chromo[-1]` and train a clean RF/XGB on that subset.


In [ ]:
# ================================
# 1. set up data and base model
# ================================

# GA will search over training features
New_FS = X_train.copy()

# labels for GA
y_trn = y_train.reset_index(drop=True)

# NOTE:
# we will create StratifiedKFold INSIDE the GA outer loop
# e.g. kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42 + run)
# so we don't fix it here

model = cuml.ensemble.RandomForestClassifier(
    n_estimators=100,  
    max_depth=12,
    max_features=1.0,  
    min_samples_leaf=2,
    bootstrap=True,
    n_bins=16,
    random_state=42,
)

# GA/global hyperparameters
size = 50
n_feat = New_FS.shape[1]


/home/zhdorothy/miniconda3/envs/rapids25/lib/python3.11/site-packages/cuml/internals/api_decorators.py:368: UserWarning: For reproducible results in Random Forest Classifier or for almost reproducible results in Random Forest Regressor, n_streams=1 is recommended. If n_streams is > 1, results may vary due to stream/thread timing differences, even when random_state is set
  return init_func(self, *args, **kwargs)


In [9]:
# ================================
# 2. population init + fitness
# ================================
def initilization_of_population(size, n_feat, off_ratio=0.3, rng=None):
    """
    create an initial list of chromosomes
    each chromosome is a boolean mask over features
    about `off_ratio` will be OFF (False), the rest ON (True), then shuffled
    """
    if rng is None:
        rng = np.random.default_rng(42)
    population = []
    off_cnt = int(off_ratio * n_feat)
    for _ in range(size):
        chromosome = np.ones(n_feat, dtype=bool)
        chromosome[:off_cnt] = False
        rng.shuffle(chromosome)
        population.append(chromosome)
    return population


def fitness_score(population, kfold=None, model=None, X=None, y=None):
    """
    evaluate each chromosome by k-fold CV accuracy (now using precomputed numpy folds)

    population: list of boolean masks
    kfold: kept for backward-compatibility (not used if global `folds` exists)
    model: sklearn classifier INSTANCE or FACTORY
           - if it's an instance, we'll clone-like recreate per fold
           - if it's a callable, we'll call it to get a fresh model
    X, y: ignored if global X_np / y_np / folds are defined
    """
    # 1) use global numpy data (fast path)
    global X_np, y_np, folds

    scores, newtp, newfp, newtn, newfn = [], [], [], [], []

    for chromosome in population:
        # boolean mask -> column indices
        col_idx = np.where(chromosome)[0]

        # empty subset guard
        if col_idx.size == 0:
            scores.append(0.0)
            newtp.append(0); newfp.append(0); newtn.append(0); newfn.append(0)
            continue

        tp_list, fp_list, tn_list, fn_list, acc_list = [], [], [], [], []

        # 2) use precomputed folds
        for train_idx, test_idx in folds:
            X_tr = X_np[train_idx][:, col_idx]
            X_te = X_np[test_idx][:, col_idx]
            y_tr = y_np[train_idx]
            y_te = y_np[test_idx]

            # make a fresh model every fold
            if callable(model):
                clf = model()
            else:
                # assume it's already an instance we can reuse safely
                # (most of the time better to pass a factory)
                clf = model

            clf.fit(X_tr, y_tr)
            preds = clf.predict(X_te)

            # confusion matrix (2-class, force labels)
            tn_i, fp_i, fn_i, tp_i = confusion_matrix(
                y_te, preds, labels=[0, 1]
            ).ravel()

            tp_list.append(tp_i); fp_list.append(fp_i)
            tn_list.append(tn_i); fn_list.append(fn_i)

            acc_list.append(accuracy_score(y_te, preds))

        # aggregate over folds
        scores.append(float(np.mean(acc_list)))
        newtp.append(int(np.sum(tp_list)))
        newfp.append(int(np.sum(fp_list)))
        newtn.append(int(np.sum(tn_list)))
        newfn.append(int(np.sum(fn_list)))

    # 3) sort by score desc
    scores = np.array(scores, dtype=float)
    population = np.array(population, dtype=object)

    total_score = scores.sum()
    if total_score == 0:
        weights = np.ones_like(scores) / len(scores)
    else:
        weights = scores / total_score

    newtp = np.array(newtp); newfp = np.array(newfp)
    newtn = np.array(newtn); newfn = np.array(newfn)

    inds = np.argsort(scores)[::-1]

    return (
        list(scores[inds]),
        list(population[inds]),
        list(weights[inds]),
        list(newtp[inds]),
        list(newfp[inds]),
        list(newtn[inds]),
        list(newfn[inds]),
    )


In [10]:
# ================================
# 3. GA operators (selection / crossover / mutation)
# ================================
def selection(pop_after_fit, weights, k):
    """
    Sample k chromosomes from the current population using
    fitness-based weights (roulette-wheel style).
    """
    picked = random.choices(pop_after_fit, weights=weights, k=k)
    return list(picked)


def crossover(p1, p2, crossover_rate):
    """
    Single-point crossover.
    Two parents → two children.
    With probability <crossover_rate> we swap tails after a random point.
    Otherwise we just copy the parents.
    """
    c1, c2 = p1.copy(), p2.copy()

    # need at least 3 genes to do a meaningful 1-point crossover
    if random.random() < crossover_rate and len(p1) > 2:
        pt = random.randint(1, len(p1) - 2)
        c1 = np.concatenate((p1[:pt], p2[pt:]))
        c2 = np.concatenate((p2[:pt], p1[pt:]))

    return [c1, c2]


def mutation(chromosome, mutation_rate):
    """
    Flip each bit with probability = mutation_rate.
    Return a NEW chromosome (don’t mutate callers' copy by surprise).
    """
    mutated = chromosome.copy()
    for i in range(len(mutated)):
        if random.random() < mutation_rate:
            mutated[i] = not mutated[i]
    return mutated


In [11]:
# ================================
# 4. full GA loop (original style)
# ================================
def generations(size, n_feat, crossover_rate, mutation_rate, n_gen):
    """
    run GA for n_gen generations in one shot
    (this is the original style)
    """
    best_chromo = []
    best_score = []

    # start with random population
    population_nextgen = initilization_of_population(size, n_feat)

    for gen in range(n_gen):
        # evaluate population
        scores, pop_after_fit, weights, tp, fp, tn, fn = fitness_score(population_nextgen)

        # best of this generation
        top_score = scores[0]
        top_chrom = pop_after_fit[0]
        print("gen", gen, "best_acc=", round(top_score, 4), "on_features=", int(top_chrom.sum()))

        # elitism (keep 2)
        elites = pop_after_fit[:2]
        k = size - 2

        # select parents
        parents = selection(pop_after_fit, weights, k)

        # make children
        children = []
        for i in range(0, len(parents), 2):
            p1 = parents[i]
            p2 = parents[(i + 1) % len(parents)]
            for child in crossover(p1, p2, crossover_rate):
                mutation(child, mutation_rate)
                children.append(child)

        # build next population
        population_nextgen = []
        for c in elites:
            population_nextgen.append(c)
        for c in children:
            if len(population_nextgen) < size:
                population_nextgen.append(c)

        # keep history
        best_chromo.append(top_chrom)
        best_score.append(top_score)

    return best_chromo, best_score


In [12]:
# ================================
# 5. single GA step (one generation)
# ================================
def ga_one_step(
    population,
    size,
    crossover_rate,
    mutation_rate,
    model,
):
    """
    Run GA for exactly 1 generation and return:
      - next_population
      - best_chromo (from current generation)
      - best_score  (from current generation)
    Uses global X_np, y_np, folds via fitness_score().
    """
    # 1) evaluate current population
    scores, pop_after_fit, weights, tp, fp, tn, fn = fitness_score(
        population,
        model=model,
        # kfold, X, y not needed anymore
    )

    # 2) best in this generation
    best_score  = scores[0]
    best_chromo = pop_after_fit[0]

    # 3) keep top-2 (elitism)
    elites = pop_after_fit[:2]
    k = size - 2

    # 4) select parents for the remaining slots
    parents = selection(pop_after_fit, weights, k)

    # 5) crossover + mutation → build children
    children = []
    for i in range(0, len(parents), 2):
        p1 = parents[i]
        p2 = parents[(i + 1) % len(parents)]  # wrap around
        for child in crossover(p1, p2, crossover_rate):
            # your mutation() mutates in-place and returns None in original code
            mutation(child, mutation_rate)
            children.append(child)

    # 6) build next population (elites first)
    next_pop = []
    # add elites
    for e in elites:
        next_pop.append(e)
    # add children until we reach "size"
    for c in children:
        if len(next_pop) < size:
            next_pop.append(c)

    return next_pop, best_chromo, best_score


# GA BEST NPZ

In [ ]:
BEST_PATH = "../data/ga_best.npz"
TOP_K = 20

def update_top_solutions_by_run(best_path, run_id, new_score, new_mask, new_cols, top_k=20):
    """
    Keep at most `top_k` best runs (one entry per run_id).
    If file exists but has old / wrong keys, we re-initialize.
    """
    prev_run_ids, prev_scores, prev_masks, prev_cols = [], [], [], []

    if os.path.exists(best_path):
        try:
            prev = np.load(best_path, allow_pickle=True)
            # try to read modern keys
            prev_run_ids = prev["run_ids"].tolist()
            prev_scores  = prev["top_scores"].tolist()
            prev_masks   = prev["top_masks"].tolist()
            prev_cols    = prev["top_cols"].tolist()
        except KeyError:
            # old / incompatible file → start fresh
            prev_run_ids, prev_scores, prev_masks, prev_cols = [], [], [], []

    # update or append
    if run_id in prev_run_ids:
        idx = prev_run_ids.index(run_id)
        if new_score > prev_scores[idx]:
            prev_scores[idx] = float(new_score)
            prev_masks[idx]  = new_mask
            prev_cols[idx]   = new_cols
    else:
        prev_run_ids.append(run_id)
        prev_scores.append(float(new_score))
        prev_masks.append(new_mask)
        prev_cols.append(new_cols)

    # keep top_k
    order = np.argsort(prev_scores)[::-1][:top_k]

    top_run_ids = [prev_run_ids[i] for i in order]
    top_scores  = [prev_scores[i]  for i in order]
    top_masks   = [prev_masks[i]   for i in order]
    top_cols    = [prev_cols[i]    for i in order]

    # save back
    np.savez(
        best_path,
        run_ids=np.array(top_run_ids, dtype=int),
        top_scores=np.array(top_scores, dtype=float),
        top_masks=np.array(top_masks, dtype=object),
        top_cols=np.array(top_cols, dtype=object),
    )
    return top_run_ids, top_scores

## Repeated GA with stratified 5-fold CV (20×)

In [14]:
N_REPEATS = 20
GENS_PER_RUN = 200

for run in range(N_REPEATS):
    # 1) No need to create a new kfold here anymore
    # kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42 + run)

    # 2) initialize population
    pop = initilization_of_population(size=size, n_feat=n_feat)
    history_chromo, history_score = [], []

    for gen in range(GENS_PER_RUN):
        pop, bc, bs = ga_one_step(
            pop,
            size=size,
            crossover_rate=0.8,
            mutation_rate=0.05,
            model=model,        # keep this (preferably pass a factory)
        )
        history_chromo.append(bc)
        history_score.append(bs)

    # pick the best solution inside THIS run
    best_idx   = int(np.argmax(history_score))
    best_score = float(history_score[best_idx])
    best_mask  = history_chromo[best_idx]
    best_cols  = np.where(best_mask)[0]

    run_ids, top_scores = update_top_solutions_by_run(
        BEST_PATH,
        run_id=run,
        new_score=best_score,
        new_mask=best_mask,
        new_cols=best_cols,
        top_k=5,
    )

    print("top runs:", run_ids)
    print("top scores:", top_scores)


top runs: [14, 3, 9, 2, 1]
top scores: [0.9254545454545454, 0.9236363636363636, 0.9072727272727272, 0.9072727272727272, 0.9054545454545455]
top runs: [14, 3, 9, 2, 1]
top scores: [0.9254545454545454, 0.9236363636363636, 0.9072727272727272, 0.9072727272727272, 0.9054545454545455]
top runs: [14, 3, 9, 2, 1]
top scores: [0.9254545454545454, 0.9236363636363636, 0.9072727272727272, 0.9072727272727272, 0.9054545454545455]
top runs: [14, 3, 9, 2, 1]
top scores: [0.9254545454545454, 0.9236363636363636, 0.9072727272727272, 0.9072727272727272, 0.9054545454545455]
top runs: [14, 3, 9, 2, 1]
top scores: [0.9254545454545454, 0.9236363636363636, 0.9072727272727272, 0.9072727272727272, 0.9054545454545455]
top runs: [14, 3, 9, 2, 1]
top scores: [0.9254545454545454, 0.9236363636363636, 0.9072727272727272, 0.9072727272727272, 0.9054545454545455]
top runs: [14, 3, 9, 2, 1]
top scores: [0.9254545454545454, 0.9236363636363636, 0.9072727272727272, 0.9072727272727272, 0.9054545454545455]
top runs: [7, 14, 3,

# Show the result

In [ ]:
best = np.load("../data/ga_best.npz", allow_pickle=True)
run_ids    = best["run_ids"].tolist()
top_scores = best["top_scores"].tolist()
top_cols   = best["top_cols"].tolist()

print("=== TOP RUNS ===")
for rid, s, cols in zip(run_ids, top_scores, top_cols):
    print(f"run {rid}: score={s:.4f}, cols(first 20)={cols[:20]}, total={len(cols)}")


=== TOP RUNS ===
run 7: score=0.9255, cols(first 5)=[0 1 2 3 4], total=20
run 14: score=0.9255, cols(first 5)=[0 1 2 3 4], total=20
run 3: score=0.9236, cols(first 5)=[0 1 3 5 6], total=30
run 9: score=0.9073, cols(first 5)=[0 1 2 3 4], total=29
run 2: score=0.9073, cols(first 5)=[1 2 4 5 6], total=26
